# **Machine Learning for Material Discovery of Hydrogen Catalysts**



### **Motivation**:
The overall motivation for this work is to accelerate the discovery of new materials that can solve the global challenge of renewable energy storage, the mechanism relies on predicting the relaxed state (structure and energy).

### **Initial Structure to Relaxed Energy (IS2RE)**
Given the initial state of a system the IS2RE task aims to predict the relaxed energy. The approach to achive this goal is by training and evaluating Graph Neural Networks (GNN). 

```--> In progress```

What is the best representation based on:

- Only distances    (GNN 1)
- Angles            (GNN 2)
- Geometry          (GNN 3)

*What is their advantage to pronosticate the Relaxed Energy among different GNN.*

**Steps:**
- Train 3 different custom GNNs 
- Compare results
- Which is the best one based on my results
- Using the best one I have to insert `attention` --> (https://github.com/gordicaleksa/pytorch-GAT/blob/main/The%20Annotated%20GAT%20(Cora).ipynb) ==> [*used with sequencial data*]
- Check final results and compare them.
- Why the final result is better or worse. 


* * *
### **Dataset information**
Each Data object includes the following information for each corresponding system (assuming K atoms):

* `sid` - [1] System ID corresponding to each structure
* `edge_index` - [2 x  J] Graph connectivity with index 0 corresponding to neighboring atoms and index 1 corresponding to center atoms. J corresponds to the total edges as determined by a nearest neighbor search.
* `atomic_numbers` - [K x 1] Atomic numbers of all atoms in the system
* `pos` - [K x 3] Initial structure positional information of all atoms in the system (x, y, z cartesian coordinates)
* `natoms` - [1] Total number atoms in the system
* `cell` -  [3  x 3] System unit cell (necessary for periodic boundary condition (PBC) calculations)
* `cell_offsets` - [J x 3] offset matrix where each index corresponds to the unit cell offset necessary to find the corresponding neighbor in  `edge_index`. For example,  `cell_offsets[0, :] = [0,1,0]` corresponds to `edge_index[:, 0]= [1,0]` representing node 1 as node 0’s neighbor located one unit cell over in the +y direction.
* `tags` - [K x 1] Atomic tag information: 0 - Fixed, sub-surface atoms, 1 - Free, surface atoms 2 - Free, adsorbate atoms

Train/Val LMDBs additionally contain the following attributes:

* `y_init` - [1] Initial structure energy of the system
* `y_relaxed` - [1] Relaxed structure energy of the system (Energy distribution)
* `pos_relaxed` - [K x 3] Relaxed structure positional information of all atoms in the system (x, y, z cartesian coordinates)


*This LMDB (Lightning Memory-Mapped Database) file requires no additional processing and is ready to be used directly with the repository’s Datasets and DataLoaders.*


***
# **SchNet (Deep learning neural network)**
## **Hyperparameter Search using Multi Head Attention (SchNet)**

### ***1 Custom Architecture (`SchNetWithMHA`)***
This class merges local physics (SchNet) with global context (Multihead Attention).

- **PBC Physics:** 
It uses the cell (the 3D bounding box of the unit cell) and `cell_offsets` (which neighboring cell the atom is in) to shift the atomic positions before calculating distance.

    - ***Theoretical Reason:*** Catalysts are infinite crystalline slabs (Periodic Boundary Conditions). An atom on the far-left edge interacts with an atom on the far-right edge of the neighboring repeating cell. Without this fix, the network thinks those atoms are far away, breaking the quantum mechanical reality of the simulation.

- **Step 1: Bridging Sparse to Dense (`to_dense_batch`):**
What it does: It takes a flat list of atoms (sparse graph) and pads them into uniform rectangular blocks (dense batches). It also creates a boolean mask (True for real atoms, False for padding).

    - ***Theoretical Reason:*** GPUs process data fastest in uniform matrix blocks. Because different molecules have different numbers of atoms (e.g., 50 vs. 75), we must pad the smaller molecules with "dummy" atoms to pass them through a standard Transformer/Attention layer.

- **Steps 2 & 3: Multihead Attention & Masking:**
What it does: Applies the nn.MultiheadAttention algorithm to the node embeddings, explicitly passing `key_padding_mask=~mask`.

    - ***Theoretical Reason:*** Standard SchNet convolutions are local (bounded by the cutoff radius). Global Attention allows atoms to "feel" changes on the complete opposite side of the molecule. The ~mask (bitwise NOT) is physically crucial: it forces the attention mathematical function to assign a probability of 0 to the "dummy" padded atoms, ensuring the network only calculates interactions between real atoms.

### ***2. The Optuna Objective Function (`objective`)***
This function dictates how the Bayesian search algorithm grades different hyperparameter combinations.

- **Step 4: Defining the Search Space:**
What it does: Uses `trial.suggest_*` to define ranges for learning rate, hidden channels, attention heads, etc.

    - ***Theoretical Reason:*** Hyperparameters dictate the memory capacity and learning stability of the network. By defining strict bounds (like making sure `hidden_channels` is perfectly divisible by `num_heads`), we prevent the optimizer from testing mathematically impossible or physically unreasonable combinations.

- **The Training and Validation Loops:**
What it does: Trains the model on the 10k dataset, then tests it on the ~23k validation dataset to calculate the `val_mae`.

    - ***Theoretical Reason:*** We must train the model so it learns the physics of that specific hyperparameter guess. We score it exclusively on the validation set to measure generalization—ensuring the model hasn't just memorized the training data (overfitting).

- **Step 5: Guided Search Pruning (`trial.should_prune`):**
What it does: Reports the validation score to Optuna at the end of every epoch. If the score is terrible compared to previous trials, it aborts the loop.

    - ***Theoretical Reason:*** GPU compute is expensive. If a hyperparameter combination proves it is failing by `epoch 3`, there is no mathematical reason to train it to `epoch 15`. The `MedianPruner` mathematically guarantees we only spend time fully training the most promising configurations.

### ***3. Execution of main function `get_hyperparameters()`***
What it does: Creates the study, runs the objective function for `n_trials`, and prints out a table of the best results.

Optuna's Tree-structured Parzen Estimator (TPE) algorithm tracks the history of all trials. It uses Bayesian probability to guess which combination of parameters will yield the lowest error next. The final table gives the exact, optimal mathematical recipe to use in the final, long-running production model.